# M3 TimeGrad — run dedicato, pensato per girare mentre dormi

Notebook **solo M3**. Fai partire **una cella** (## 7) che allena e campiona, poi puoi allontanarti. Quella cella, **alla fine**, stampa un **BLOCCO DI RECUPERO** con *tutto* il necessario (riga CSV grezza a piena precisione, versioni, git, tempi): se la VM si ricicla mentre dormi, basta incollarmi quel blocco e ricostruisco **senza perdere nulla**.

> ⚠️ **Verita` sul free tier (leggi!):** su Colab gratis, se **chiudi il tab** o il **laptop va in sospensione**, il run viene **ucciso** dopo ~90 min (niente background execution; TimeGrad non ha resume). Un run di 2-3h **incustodito puo` NON finire**.
>
> **Per massimizzare le chance di arrivare in fondo mentre dormi:**
> 1. **Monta Drive** (cella ## 5): i risultati finiscono nel TUO Drive e **sopravvivono** anche se la VM muore dopo aver finito.
> 2. Tieni il **laptop sveglio e collegato** (alimentatore + WiFi stabile). Su Mac, in un Terminale: `caffeinate -dimsu` (lo schermo puo` spegnersi, il sistema NO). **Non chiudere il coperchio** se questo manda in sospensione.
> 3. Lascia **il tab Colab aperto**.
>
> Anche cosi` il free puo` staccarti: il blocco di recupero + il backup su Drive sono l'assicurazione.

## 1 - Verifica la GPU
Se vedi una **Tesla T4** (o simile) la GPU e` attiva. Altrimenti: **Runtime > Change runtime type > GPU**, poi riesegui.

In [ ]:
!nvidia-smi

## 2 - Scarica il codice del progetto
Clona il repo sul branch `feature/port-ladder-exchange` (contiene gia` il fix pandas<2.2).

In [ ]:
REPO_URL = "https://github.com/Icaica14/pml-diffusion-tsf.git"
BRANCH   = "feature/port-ladder-exchange"

import os
if not os.path.isdir("/content/pml-diffusion-tsf"):
    !git clone --branch $BRANCH $REPO_URL /content/pml-diffusion-tsf
%cd /content/pml-diffusion-tsf
!git log --oneline -1

## 3 - Installa le librerie pesanti (versioni bloccate)
**GluonTS 0.13** + **PyTorchTS** + il PyTorch CUDA di Colab. Downgrade di **NumPy < 2** e **pandas < 2.2** (servono a GluonTS 0.13).

A fine cella: **Runtime > Restart session**, poi **riparti dalla cella 4** (NON rifare questa).

In [ ]:
!pip install -q "gluonts[torch]==0.13.7" "numpy<2" "pandas<2.2"
!pip install -q --no-deps "git+https://github.com/zalandoresearch/pytorch-ts.git@81be06bcc"
print("\nInstallazione finita. Ora: Runtime > Restart session, poi continua dalla cella 4.")

## 4 - Controllo ambiente (dopo il restart)
Se arriva in fondo senza errori e vedi `cuda disponibile: True`, sei pronto.

In [ ]:
%cd /content/pml-diffusion-tsf
import numpy, pandas, torch, gluonts, pts
print("numpy   :", numpy.__version__)
print("pandas  :", pandas.__version__)
print("torch   :", torch.__version__)
print("gluonts :", gluonts.__version__)
print("cuda disponibile:", torch.cuda.is_available())
print("device  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5 - (consigliato per la notte) Monta Google Drive
Serve a **salvare i risultati nel tuo Drive**, cosi` sopravvivono al riciclo della VM. Parte un **popup di autorizzazione**: accettalo **prima** di andare a dormire. Se salti questo passo, resta comunque il BLOCCO DI RECUPERO stampato a video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montato. Backup -> /content/drive/MyDrive/pml_m3_backup/")

## 6 - Scegli il dataset
`"electricity"` e` il run vero (D=321, ~2-3h). `"exchange"` per una prova veloce.

In [ ]:
DATASET = "electricity"        # "exchange" per un giro veloce di prova
CONFIG  = f"configs/data_{DATASET}.yaml"
CHUNK   = 256 if DATASET == "electricity" else 0
EPOCHS  = 50 if DATASET == "electricity" else 20
print(f"dataset={DATASET}  config={CONFIG}  chunk={CHUNK}  epochs={EPOCHS}")

## 7 - RUN (training + predict) + BLOCCO DI RECUPERO  ←  la cella lunga
Fai partire **questa** e poi puoi allontanarti. Allena TimeGrad, campiona, e **alla fine stampa il blocco di recupero** (riga CSV a piena precisione + versioni + git + tempi) e fa il **backup su Drive** se montato. Quando ti svegli: se sei ancora connesso vai alla cella ## 8; se la VM e` morta, **copia tutto il blocco di recupero** e incollamelo.

In [ ]:
%cd /content/pml-diffusion-tsf
import datetime, pathlib, subprocess
_start = datetime.datetime.now(datetime.timezone.utc)
print("START (UTC):", _start.isoformat())

# ---- TRAINING + PREDICT (il pezzo lungo: ~2-3h su Electricity) ----
!python -m experiments.run_timegrad --config $CONFIG --chunk $CHUNK --epochs $EPOCHS --device cuda

# ---- BLOCCO DI RECUPERO: stampa tutto per ricostruire/recuperare ----
_end = datetime.datetime.now(datetime.timezone.utc)

def _sh(c):
    try:
        return subprocess.run(c, capture_output=True, text=True, shell=True).stdout.strip()
    except Exception as e:
        return "(errore: " + str(e) + ")"

B = "#" * 72
print("\n\n" + B)
print("# BLOCCO DI RECUPERO M3 -- se la VM si ricicla, INCOLLA TUTTO QUESTO a Claude")
print(B)
print("# inizio (UTC):", _start.isoformat())
print("# fine   (UTC):", _end.isoformat())
print("# durata (min):", round((_end - _start).total_seconds() / 60, 1))
print("# dataset:", DATASET, "| config:", CONFIG, "| chunk:", CHUNK, "| epochs:", EPOCHS)
print("# git HEAD:", _sh("git rev-parse HEAD"))
print("# git log :", _sh("git log --oneline -1"))
try:
    import numpy, pandas, torch, gluonts, pts
    print("# versioni: numpy", numpy.__version__, "| pandas", pandas.__version__,
          "| torch", torch.__version__, "| gluonts", gluonts.__version__)
except Exception as e:
    print("# versioni: (errore import:", e, ")")
print("# " + "-" * 70)
print("# REGISTRY (CSV grezzo, piena precisione) -- header + righe timegrad:")
reg = pathlib.Path("/content/pml-diffusion-tsf/results/registry.csv")
if reg.is_file():
    _lines = reg.read_text().splitlines()
    print(_lines[0])
    _new = [_l for _l in _lines[1:] if len(_l.split(",")) > 2 and _l.split(",")[2] == "timegrad"]
    for _l in _new:
        print(_l)
    print("# (" + str(len(_new)) + " righe timegrad)")
    _out = "\n".join([_lines[0]] + _new) + "\n"
    pathlib.Path("/content/pml-diffusion-tsf/results/colab_new_rows.csv").write_text(_out)
    if pathlib.Path("/content/drive/MyDrive").is_dir():
        _bk = pathlib.Path("/content/drive/MyDrive/pml_m3_backup")
        _bk.mkdir(parents=True, exist_ok=True)
        (_bk / "registry.csv").write_text(reg.read_text())
        (_bk / "colab_new_rows.csv").write_text(_out)
        print("# BACKUP su Drive OK: /content/drive/MyDrive/pml_m3_backup/")
    else:
        print("# (Drive non montato: nessun backup su Drive)")
else:
    print("# !! results/registry.csv NON trovata: il run e' forse fallito prima di scrivere.")
print(B)
print("# FINE BLOCCO DI RECUPERO")
print(B)

## 8 - Porta a casa i risultati (se sei sveglio e connesso)
Scarica il CSV con le righe timegrad. (Se hai montato Drive, una copia e` gia` li`.)

In [ ]:
from google.colab import files
files.download("/content/pml-diffusion-tsf/results/colab_new_rows.csv")

## 9 - E adesso?
Mandami il `colab_new_rows.csv` **oppure** incolla il BLOCCO DI RECUPERO della cella ## 7. Unisco la riga `timegrad` Electricity alla registry e committo io in locale. Se invece il run si e` interrotto a meta` (guarda a che epoca e` arrivato nel log), lo rilanciamo — niente resume, purtroppo.